# Scenery inspection — DroneWaste + AerialWaste

Review each image with its agent-generated scenery description and GT waste labels. Reads the per-chunk JSONLs under `data/captions/scenery_chunks/` (resume-safe, no need to merge first), joins with `data/vqa/split_index.jsonl` for split + GT metadata, and provides one-at-a-time and grid viewers.

In [4]:
import json, random, textwrap
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

SCENERY_CHUNKS = Path('/home/ids/diecidue/data/captions/scenery_chunks')
SCENERY_MERGED = Path('/home/ids/diecidue/data/captions/scenery.jsonl')
SPLIT_INDEX    = Path('/home/ids/diecidue/data/vqa/split_index.jsonl')

# ---- load split_index (GT labels + dataset + split per image) ----
meta_by_path = {}
for line in SPLIT_INDEX.read_text().splitlines():
    if not line.strip():
        continue
    r = json.loads(line)
    meta_by_path[r['image_path']] = r

# ---- load every scenery line (chunks first, then merged if present) ----
rows = []
seen_paths = set()
files = sorted(SCENERY_CHUNKS.glob('chunk_*.jsonl'))
if SCENERY_MERGED.exists():
    files = [SCENERY_MERGED] + files  # merged takes precedence on dedup
for f in files:
    for line in f.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            d = json.loads(line)
        except Exception:
            continue
        p = d.get('image_path')
        if not p or p in seen_paths:
            continue
        seen_paths.add(p)
        m = meta_by_path.get(p, {})
        rows.append({
            'image_path': p,
            'scenery':    d.get('scenery', ''),
            'model':      d.get('model', ''),
            'chunk':      f.name,
            'dataset':    m.get('dataset', '?'),
            'image_id':   m.get('image_id', '?'),
            'split':      m.get('split', '?'),
            'gt_categories': m.get('gt_categories', []),
        })

errors  = [r for r in rows if r['scenery'].startswith('ERROR:')]
good    = [r for r in rows if not r['scenery'].startswith('ERROR:')]
print(f'total scenery lines        : {len(rows):,}')
print(f'  good                     : {len(good):,}')
print(f'  errors                   : {len(errors):,}')
print(f'distinct chunks            : {len({r["chunk"] for r in rows}):,}')
print()
print('by dataset / split:')
for k, n in sorted(Counter((r["dataset"], r["split"]) for r in good).items()):
    print(f'  {k[0]:<22} {k[1]:<5} {n:>6}')

total scenery lines        : 2,465
  good                     : 2,464
  errors                   : 1
distinct chunks            : 83

by dataset / split:
  aerialwaste_m2         test       1
  aerialwaste_m2         train      1
  dronewaste_paper10     test     378
  dronewaste_paper10     train   1718
  dronewaste_paper10     val      366


In [5]:
# ---- choose what to inspect (edit, then re-run this cell + the viewer) ----
DATASET   = None      # 'aerialwaste_m2' | 'dronewaste_paper10' | None for all
SPLIT     = None      # 'train' | 'val' | 'test' | None for all
POS_ONLY  = False     # True -> only images with GT waste labels
NEG_ONLY  = False     # True -> only images with NO GT waste labels
CHUNK     = None      # e.g. 'chunk_0042.jsonl' | None for all
SAMPLE_N  = None      # None for all matches; otherwise random subsample of this size
SEED      = 0

FORBIDDEN = ('waste','debris','garbage','dumping','rubble','scrap','litter','pollution')
AUDIT_VIOLATIONS_ONLY = False   # True -> only show rows that contain a forbidden word

view = good[:]
if DATASET:  view = [r for r in view if r['dataset'] == DATASET]
if SPLIT:    view = [r for r in view if r['split']   == SPLIT]
if POS_ONLY: view = [r for r in view if r['gt_categories']]
if NEG_ONLY: view = [r for r in view if not r['gt_categories']]
if CHUNK:    view = [r for r in view if r['chunk']   == CHUNK]
if AUDIT_VIOLATIONS_ONLY:
    view = [r for r in view if any(w in r['scenery'].lower() for w in FORBIDDEN)]
if SAMPLE_N and len(view) > SAMPLE_N:
    random.Random(SEED).shuffle(view)
    view = view[:SAMPLE_N]
print(f'view: {len(view):,} rows')

view: 2,464 rows


In [6]:
# ---- one-at-a-time viewer (Prev / Next buttons) ----
from ipywidgets import Button, HBox, VBox, Output, HTML, Label
from IPython.display import display, clear_output

def _highlight_forbidden(text):
    return [w for w in FORBIDDEN if w in text.lower()]

state = {'i': 0}
out = Output()
header = HTML()

def _render():
    if not view:
        with out:
            clear_output(wait=True)
            print('empty view — adjust the filters in the previous cell')
        header.value = ''
        return
    i = state['i']
    r = view[i]
    hits = _highlight_forbidden(r['scenery'])
    flag = f"  <span style='color:#c00'><b>FORBIDDEN HITS: {hits}</b></span>" if hits else ''
    gt   = ', '.join(r['gt_categories']) if r['gt_categories'] else '(negative)'
    header.value = (
        f"<b>[{i+1}/{len(view)}]</b>  {r['dataset']} | split={r['split']} | "
        f"{r['chunk']} | id={r['image_id']}{flag}<br>"
        f"<b>GT waste labels:</b> {gt}<br>"
        f"<code>{r['image_path']}</code>"
    )
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 8))
        try:
            ax.imshow(Image.open(r['image_path']))
        except Exception as e:
            ax.text(0.5, 0.5, f'image load failed:\n{e}', ha='center', va='center')
        ax.axis('off')
        plt.show()
        print(textwrap.fill(r['scenery'], width=110))

def _step(delta):
    if not view:
        return
    state['i'] = (state['i'] + delta) % len(view)
    _render()

btn_prev = Button(description='◀ Prev', layout={'width': '100px'})
btn_next = Button(description='Next ▶', layout={'width': '100px'})
btn_prev.on_click(lambda b: _step(-1))
btn_next.on_click(lambda b: _step(+1))

display(VBox([HBox([btn_prev, btn_next]), header, out]))
_render()

In [ ]:
# ---- grid scan (image-only, scenery shown as caption under each cell) ----
def show_grid(start=0, n=12, cols=3, max_side=320, wrap=44):
    sub = view[start:start + n]
    if not sub:
        print('empty view')
        return
    rows_ = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 4.5, rows_ * 5.2))
    axes = axes.flatten() if rows_ * cols > 1 else [axes]
    for ax, r in zip(axes, sub):
        try:
            img = Image.open(r['image_path'])
            w, h = img.size
            scale = max_side / max(w, h) if max(w, h) > max_side else 1
            img = img.resize((int(w * scale), int(h * scale)))
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f'fail: {e}', ha='center', va='center')
        ax.axis('off')
        hits = _highlight_forbidden(r['scenery'])
        flag = '  [!]' if hits else ''
        head = f'{r["dataset"][:2].upper()} | {r["split"]} | id={r["image_id"]}{flag}'
        ax.set_title(head + '\n' + textwrap.fill(r['scenery'], width=wrap), fontsize=8)
    for ax in axes[len(sub):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

show_grid(start=0, n=12, cols=3)

In [ ]:
# ---- forbidden-word audit across the whole good set (no GUI) ----
violations = []
for r in good:
    hits = _highlight_forbidden(r['scenery'])
    if hits:
        violations.append((r, hits))
print(f'forbidden-word violations: {len(violations)} / {len(good)}')
for r, hits in violations[:25]:
    print(f'  {r["chunk"]} | {r["dataset"]:<22} | id={r["image_id"]:<8} | {hits}')
    print(f'    {r["scenery"]}')
    print()

In [ ]:
# ---- search captions by substring ----
QUERY = 'river'
matches = [r for r in good if QUERY.lower() in r['scenery'].lower()]
print(f'"{QUERY}" matches: {len(matches)}')
for r in matches[:5]:
    print(f'  {r["dataset"]} | {r["split"]} | id={r["image_id"]}')
    print(f'    {r["scenery"]}')
    print()